In [47]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

In [10]:
BINO_PRED_PATH = "../../data/binoculars/42rand_state-1000-samples/combined.csv"
STYLOMETRY_PRED_PATH = "../../data/stylometry/stylometry-2000.csv"


In [37]:
bino_df = pd.read_csv(BINO_PRED_PATH)
stylo_df = pd.read_csv(STYLOMETRY_PRED_PATH)
invalid_rows = bino_df[pd.to_numeric(bino_df['binoculars_score'], errors='coerce').isna()]
invalid_rows
stylo_df = stylo_df[~stylo_df["Unnamed: 0"].isin(invalid_rows["Unnamed: 0"])]
bino_df = bino_df[~bino_df["Unnamed: 0"].isin(invalid_rows["Unnamed: 0"])]
stylo_df.set_index("Unnamed: 0")
bino_df.set_index("Unnamed: 0")
stylo_df_filtered = stylo_df.loc[stylo_df.index.isin(bino_df.index)]

# Get rows in bino_df that have indices NOT in stylo_df
bino_df_filtered = bino_df.loc[bino_df.index.isin(stylo_df.index)]


In [38]:
bino_df_filtered["binoculars_score"] = bino_df_filtered["binoculars_score"].apply(pd.to_numeric)

C:\Users\boyan.bogdanov\AppData\Local\Temp\ipykernel_33776\1510869091.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bino_df_filtered["binoculars_score"] = bino_df_filtered["binoculars_score"].apply(pd.to_numeric)


In [39]:
BINOCULARS_ACCURACY_THRESHOLD = 0.9015310749276843  # optimized for f1-score
BINOCULARS_FPR_THRESHOLD = 0.8536432310785527

bino_df_filtered["prediction_accuracy"] = np.where(bino_df_filtered["binoculars_score"] > BINOCULARS_ACCURACY_THRESHOLD, 1, 0)
bino_df_filtered["prediction_fpr"] = np.where(bino_df_filtered["binoculars_score"] > BINOCULARS_FPR_THRESHOLD, 1, 0)


C:\Users\boyan.bogdanov\AppData\Local\Temp\ipykernel_33776\1938721872.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bino_df_filtered["prediction_accuracy"] = np.where(bino_df_filtered["binoculars_score"] > BINOCULARS_ACCURACY_THRESHOLD, 1, 0)
C:\Users\boyan.bogdanov\AppData\Local\Temp\ipykernel_33776\1938721872.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bino_df_filtered["prediction_fpr"] = np.where(bino_df_filtered["binoculars_score"] > BINOCULARS_FPR_THRESHOLD, 1, 0)


In [50]:
acc = np.mean(bino_df_filtered["prediction_accuracy"] == stylo_df_filtered["prediction"])
print(f"Bino accuracy & stylometry agreement: {acc}")

Bino accuracy & stylometry agreement: 0.6572601334017445


In [49]:
fpr = np.mean(bino_df_filtered["prediction_fpr"] == stylo_df_filtered["prediction"])
print(f"Bino fpr & stylometry agreement: {acc}")

Bino fpr & stylometry agreement: 0.6572601334017445


In [46]:
y_test = bino_df_filtered["is_llm"]
y_test

0       1
1       1
2       1
3       1
4       1
       ..
1975    0
1976    0
1977    0
1979    0
1980    0
Name: is_llm, Length: 1949, dtype: int64

In [48]:
errors_binoculars = (bino_df_filtered["prediction_accuracy"] != y_test).astype(int)
errors_stylo = (stylo_df_filtered["prediction"] != y_test).astype(int)

# Compute correlation between errors
corr_stylo_binoculars, _ = pearsonr(errors_stylo, errors_binoculars)

print(f"Error Correlation (Stylometry vs Binoculars): {corr_stylo_binoculars:.2f}")

Error Correlation (Stylometry vs Binoculars): 0.33


In [ ]:
def compare_predictions(stylo_df, bino_df):
    return np.mean(stylo_df["prediction"] == bino_df["prediction"])

In [ ]:
def compare_error_correlation():
    pass